# 🚀 Attrition Prediction — Model Builder
This notebook contains the **entire** pipeline in one file:
1. Install dependencies
2. Configuration
3. Check Cache (Ask to rebuild if data exists)
4. Data loading & target construction
5. Feature engineering (30 features)
6. Model training (XGBoost)
7. Evaluation (AUC, Recall, Precision)
8. Save model & featured data for the Dashboard notebook

---
## 0. Install Dependencies

In [ ]:
!pip install pandas numpy xgboost scikit-learn imbalanced-learn joblib matplotlib seaborn pyarrow python-dateutil

---
## 1. Configuration
All paths, mappings, and feature lists. **Edit paths below if your folder structure is different.**

In [ ]:
import os, sys, glob, json, time
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from dateutil.relativedelta import relativedelta
from IPython.display import display, HTML

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 60)
plt.style.use('ggplot')

# ===================== EDIT THESE PATHS =====================
BASE_DIR = os.getcwd()
PATH_HEADCOUNT   = os.path.join(BASE_DIR, '..', '01_Raw_Headcount')
PATH_PERFORMANCE = os.path.join(BASE_DIR, '..', '02_Raw_Performance')
PATH_PROMOTIONS  = os.path.join(BASE_DIR, '..', '03_Raw_Promotion')
PATH_OUTPUT      = os.path.join(BASE_DIR, '..', '04_Master_Data')
PATH_MODEL_OUTPUT = os.path.join(BASE_DIR, 'model_artifacts')
# =============================================================

os.makedirs(PATH_OUTPUT, exist_ok=True)
os.makedirs(PATH_MODEL_OUTPUT, exist_ok=True)

PREDICTION_HORIZON_MONTHS = 6
TARGET_COLUMN = 'Will_Leave_Within_Horizon'

SKIP_HEADCOUNT   = 3
SKIP_PERFORMANCE = 1
SKIP_PROMOTIONS  = 2

LEAKAGE_COLUMNS = [
    'HC_Future_Term_Current Year_EE_Vol',
    'HC_Future_Term_Current Year_EE_Invol',
    'HC_Future_Term_Current Year_EE_Other',
    'HC_Future_Term_EE_Current_Year_1_0',
    'HC_Future_Term_EE',
    'Will_Churn_Eventually', 'Is_Leaver_Row',
    'Termination Date', 'Termination Month Number',
    'Termination Reason Category', 'Termination Reason',
]

PERFORMANCE_MAP = {
    'Significantly Outperformed': 5, 'Exceeding': 5,
    'Outperformed': 4, 'Strongly Performed': 3, 'Tracking': 3,
    'Achieved Most Expectations': 2, 'Needs Improvement': 1,
    'New Hire: Needs Improvement': 1,
}

MANAGEMENT_LEVEL_MAP = {
    'Admin Business Coordinator': 'Admin', 'Admin Business Lead': 'Admin',
    'Admin Business Partner': 'Admin', 'Administrative Assistant': 'Admin',
    'Analyst': 'Analyst', 'Associate': 'Associate',
    'CEO-Chairman': 'MD+', 'Contingent Worker': 'Contingent Worker',
    'Director': 'Director', 'Employee Fixed Term': 'Admin',
    'Executive Assistant': 'Admin', 'Fund Partner': 'MD+',
    'Intern': 'Intern', 'Managing Director': 'MD+', 'Partner': 'MD+',
    'President': 'MD+', 'Principal': 'Director', 'Senior Advisor': 'MD+',
    'Senior Associate': 'Associate', 'Senior Managing Director': 'MD+',
    'Specialist': 'Specialist', 'Vice Chairman': 'MD+',
    'Vice President': 'Vice President', 'Executive Director': 'Director',
}

LEVEL_BENCHMARKS = {
    'Analyst': 2.0, 'Associate': 3.0, 'Vice President': 4.0,
    'Director': 4.0, 'MD+': 5.0, 'Admin': 5.0, 'Specialist': 3.0,
}

BASE_FEATURES = [
    'Total_Tenure_Years', 'Time_In_Title_Current', 'Cycles_In_Level',
    'Is_Stuck_In_Level', 'Days_Since_Last_Promotion', 'Consecutive_Low_Perf_Years',
    'Performance_Score', 'Is_High_Performer', 'Performance_Drop_Flag',
    'Perf_Score_Rolling_Avg_3Yr', 'Perf_Trajectory_Slope',
    'Peer_Promo_Rate', 'Peer_Pressure_Flag',
    'Manager_Past_Churn_Count', 'Manager_Performance_Score',
    'Performer_vs_Mgr_Score', 'Manager_Changed_Recently', 'Manager_Span_Of_Control',
    'Job_Family_Churn_Rate', 'Org_Level_Churn_Rate',
    'Is_Lateral_Hire', 'Is_Campus_Hire', 'Has_International_Assignment',
    'Is_Acquisition_Employee', 'FTE_Value', 'Is_Reorg_Affected', 'Is_Transfer_Recent',
    'Month_Sin', 'Month_Cos', 'Is_Post_Bonus_Window',
]
CATEGORICAL_FEATURES = ['Management_Level_Agg']

print('✅ Configuration loaded.')
print(f'   Prediction Horizon: {PREDICTION_HORIZON_MONTHS} months')
print(f'   Headcount path: {PATH_HEADCOUNT}')
print(f'   Output path: {PATH_OUTPUT}')

---
## 2. Check Cache
If the final data and model already exist, asks if you want to rebuild or skip.

In [ ]:
data_path = os.path.join(PATH_OUTPUT, 'Model_Ready_Data.parquet')
if not os.path.exists(data_path):
    data_path = os.path.join(PATH_OUTPUT, 'Model_Ready_Data.csv')

model_path = os.path.join(PATH_MODEL_OUTPUT, 'xgboost_model.joblib')

rebuild_all = True
if os.path.exists(data_path) and os.path.exists(model_path):
    print(f'\n✅ FOUND EXISTING ARTIFACTS:')
    print(f'   - Data:  {data_path}')
    print(f'   - Model: {model_path}\n')
    ans = input('Do you want to re-process data and rebuild the model? (y/n) [n]: ').strip().lower()
    if ans in ['n', '', 'no']:
        rebuild_all = False
        print('\n⏭️ SKIP: You chose not to rebuild. You can go straight to the Dashboard Notebook!')
    else:
        print('\n🔄 REBUILD: Will process all data from scratch.')
else:
    print('\n🔨 No existing cached data or model found. Proceeding with full build.')


---
## 3. Data Loading & Target Construction
Loads headcount, performance, and promotion CSVs. Builds the leakage-safe voluntary attrition target.

In [ ]:
if rebuild_all:
    def clean_id(val):
        s = str(val).strip()
        if s.endswith('.0'):
            s = s[:-2]
        return s
    
    def load_folder_csvs(folder_path, header_row_index):
        print(f'   -> Scanning: {folder_path}')
        files = glob.glob(os.path.join(folder_path, '*.csv'))
        if not files:
            print(f'   -> WARNING: No CSV files found in {folder_path}')
            return pd.DataFrame()
        df_list = []
        for f in files:
            try:
                df = pd.read_csv(f, header=header_row_index, low_memory=False)
                df_list.append(df)
                print(f'      Loaded: {os.path.basename(f)} ({len(df)} rows)')
            except Exception as e:
                print(f'      ERROR reading {os.path.basename(f)}: {e}')
        if not df_list:
            return pd.DataFrame()
        combined = pd.concat(df_list, ignore_index=True)
        print(f'   -> Total rows loaded: {len(combined)}')
        return combined
    
    print('✅ Helper functions defined.')
else:
    print('⏭️ Skipped (Cache active)')

In [ ]:
if rebuild_all:
    # ========== A. LOAD HEADCOUNT ==========
    print('=' * 60)
    print('   STEP 1: PROCESSING HEADCOUNT')
    print('=' * 60)
    
    df_hc = load_folder_csvs(PATH_HEADCOUNT, SKIP_HEADCOUNT)
    df_hc.columns = df_hc.columns.str.strip()
    
    df_hc.rename(columns={
        'Data As Of': 'Snapshot_Date',
        'Employee ID/Position ID': 'Employee_ID',
    }, inplace=True)
    
    df_hc['Snapshot_Date'] = pd.to_datetime(df_hc['Snapshot_Date'], errors='coerce')
    mask_bad = df_hc['Snapshot_Date'].isna()
    if mask_bad.any():
        raw = df_hc.loc[mask_bad, 'Snapshot_Date']
        df_hc.loc[mask_bad, 'Snapshot_Date'] = pd.to_datetime(
            pd.to_numeric(raw, errors='coerce'), unit='D', origin='1899-12-30'
        )
    df_hc['Snapshot_Date'] = df_hc['Snapshot_Date'].astype('datetime64[ms]')
    df_hc = df_hc.dropna(subset=['Snapshot_Date'])
    df_hc['Year'] = df_hc['Snapshot_Date'].dt.year
    df_hc['Month'] = df_hc['Snapshot_Date'].dt.month
    df_hc['Employee_ID'] = df_hc['Employee_ID'].apply(clean_id)
    
    # Normalize flags
    flag_cols = ['HC_LVO_EE','HC_LIN_EE','HC_LOT_EE','HC_CTT_EE','HC_BSL_EE',
                 'HC_RGI_EE','HC_RGO_EE','HC_RTT_EE','HC_XTT_EE','HC_XOT_EE',
                 'HC_XIN_EE','HC_HIR_EE','HC_HTT_EE']
    for col in flag_cols:
        if col in df_hc.columns:
            df_hc[col] = pd.to_numeric(df_hc[col], errors='coerce').fillna(0).astype(int).abs()
    
    print(f'   Voluntary leaver rows: {df_hc["HC_LVO_EE"].sum():,}')
    print(f'   Total rows: {len(df_hc):,}')
else:
    print('⏭️ Skipped (Cache active)')

In [ ]:
if rebuild_all:
    # ========== B. LOAD PERFORMANCE ==========
    print('=' * 60)
    print('   STEP 2: PROCESSING PERFORMANCE')
    print('=' * 60)
    
    df_perf_raw = load_folder_csvs(PATH_PERFORMANCE, SKIP_PERFORMANCE)
    if not df_perf_raw.empty:
        df_perf_raw.columns = df_perf_raw.columns.str.strip()
        df_perf_raw.rename(columns={'Employee ID': 'Employee_ID'}, inplace=True)
        yer_cols = [c for c in df_perf_raw.columns if 'YER Rating' in c]
        if yer_cols:
            df_perf_long = pd.melt(
                df_perf_raw, id_vars=['Employee_ID'], value_vars=yer_cols,
                var_name='Rating_Source_Col', value_name='Performance_Text'
            )
            df_perf_long['Performance_Year'] = df_perf_long['Rating_Source_Col'].str.extract(r'(\d{4})').astype(float)
            df_perf_long['Effective_Year'] = df_perf_long['Performance_Year'] + 1
            df_perf_long['Performance_Score'] = df_perf_long['Performance_Text'].str.strip().map(PERFORMANCE_MAP)
            df_perf_long['Employee_ID'] = df_perf_long['Employee_ID'].apply(clean_id)
            df_hc = pd.merge(
                df_hc,
                df_perf_long[['Employee_ID','Effective_Year','Performance_Text','Performance_Score']],
                left_on=['Employee_ID','Year'], right_on=['Employee_ID','Effective_Year'], how='left'
            )
            df_hc.drop(columns=['Effective_Year'], inplace=True, errors='ignore')
            df_hc = df_hc.sort_values(by=['Employee_ID','Snapshot_Date'])
            df_hc['Performance_Text'] = df_hc.groupby('Employee_ID')['Performance_Text'].ffill()
            df_hc['Performance_Score'] = df_hc.groupby('Employee_ID')['Performance_Score'].ffill()
            print('   -> Performance merged and forward-filled.')
    else:
        df_hc['Performance_Text'] = np.nan
        df_hc['Performance_Score'] = np.nan
else:
    print('⏭️ Skipped (Cache active)')

In [ ]:
if rebuild_all:
    # ========== C. LOAD PROMOTIONS ==========
    print('=' * 60)
    print('   STEP 3: PROCESSING PROMOTIONS')
    print('=' * 60)
    
    df_promo = load_folder_csvs(PATH_PROMOTIONS, SKIP_PROMOTIONS)
    if not df_promo.empty:
        df_promo.columns = df_promo.columns.str.strip()
        if 'Employee' in df_promo.columns:
            df_promo.rename(columns={'Employee': 'Employee_ID'}, inplace=True)
        elif 'Employee ID' in df_promo.columns:
            df_promo.rename(columns={'Employee ID': 'Employee_ID'}, inplace=True)
        df_promo.rename(columns={
            'Process Year': 'Process_Year',
            'Nomination Status - After Memo': 'Nomination_Status',
        }, inplace=True)
        required = ['Nomination_Status', 'Process_Year', 'Employee_ID']
        if all(c in df_promo.columns for c in required):
            df_promo['Is_Promoted'] = df_promo['Nomination_Status'].astype(str).str.contains('Nominated', case=False, na=False).astype(int)
            df_promo['Process_Year'] = pd.to_numeric(df_promo['Process_Year'], errors='coerce')
            df_promo['Effective_Year'] = df_promo['Process_Year'] + 1
            df_promo['Employee_ID'] = df_promo['Employee_ID'].apply(clean_id)
            df_promo = df_promo.dropna(subset=['Effective_Year','Employee_ID'])
            promo_dedup = df_promo.groupby(['Employee_ID','Effective_Year'])['Is_Promoted'].max().reset_index()
            df_hc = pd.merge(df_hc, promo_dedup, left_on=['Employee_ID','Year'],
                             right_on=['Employee_ID','Effective_Year'], how='left')
            df_hc.drop(columns=['Effective_Year'], inplace=True, errors='ignore')
            df_hc['Is_Promoted'] = df_hc['Is_Promoted'].fillna(0).astype(int)
            print('   -> Promotions merged.')
        else:
            df_hc['Is_Promoted'] = 0
    else:
        df_hc['Is_Promoted'] = 0
else:
    print('⏭️ Skipped (Cache active)')

In [ ]:
if rebuild_all:
    # ========== D. MANAGEMENT LEVEL AGGREGATION ==========
    if 'Management Level' in df_hc.columns:
        df_hc['Management_Level_Agg'] = (
            df_hc['Management Level'].astype(str).str.strip()
            .map(MANAGEMENT_LEVEL_MAP).fillna('Unmapped')
        )
    else:
        df_hc['Management_Level_Agg'] = 'Unmapped'
    
    # ========== E. BUILD LEAKAGE-SAFE TARGET ==========
    print('\n' + '=' * 60)
    print(f'   STEP 4: BUILDING TARGET (Horizon={PREDICTION_HORIZON_MONTHS}m)')
    print('=' * 60)
    
    df_hc = df_hc.sort_values(by=['Employee_ID','Snapshot_Date'])
    
    # Find first voluntary departure date per employee
    vol_leavers = df_hc[df_hc['HC_LVO_EE'] == 1].copy()
    first_leave = vol_leavers.groupby('Employee_ID')['Snapshot_Date'].min().reset_index()
    first_leave.columns = ['Employee_ID', 'Vol_Departure_Date']
    print(f'   Unique voluntary leavers: {len(first_leave):,}')
    
    df_hc = pd.merge(df_hc, first_leave, on='Employee_ID', how='left')
    
    df_hc['Horizon_End'] = df_hc['Snapshot_Date'].apply(
        lambda d: d + relativedelta(months=PREDICTION_HORIZON_MONTHS)
    )
    df_hc[TARGET_COLUMN] = (
        (df_hc['Vol_Departure_Date'].notna()) &
        (df_hc['Vol_Departure_Date'] > df_hc['Snapshot_Date']) &
        (df_hc['Vol_Departure_Date'] <= df_hc['Horizon_End'])
    ).astype(int)
    
    # Keep only ACTIVE employees
    active_mask = (
        (df_hc['HC_CTT_EE'] == 1) &
        (df_hc['HC_LVO_EE'] == 0) &
        (df_hc['Management_Level_Agg'] != 'Contingent Worker')
    )
    df = df_hc[active_mask].copy()
    df.drop(columns=['Vol_Departure_Date','Horizon_End'], inplace=True, errors='ignore')
    
    # Drop leakage columns
    leak_drop = [c for c in LEAKAGE_COLUMNS if c in df.columns]
    df.drop(columns=leak_drop, inplace=True, errors='ignore')
    
    print(f'   Active rows: {len(df):,}')
    print(f'   Target=1: {df[TARGET_COLUMN].sum():,}')
    print(f'   Target rate: {df[TARGET_COLUMN].mean():.2%}')
    print(f'   Leakage columns dropped: {len(leak_drop)}')
else:
    print('⏭️ Skipped (Cache active)')

---
## 4. Feature Engineering (30 Features)

In [ ]:
if rebuild_all:
    print('   [1/6] Tenure & Career features...')
    
    # Total Tenure
    if 'Length of Service' in df.columns:
        df['Total_Tenure_Years'] = pd.to_numeric(df['Length of Service'], errors='coerce')
        df['Total_Tenure_Years'] = df['Total_Tenure_Years'].fillna(df['Total_Tenure_Years'].median())
    else:
        df['Total_Tenure_Years'] = 0
    
    # Time in Title
    if 'Time In Title (Current)' in df.columns:
        df['Time_In_Title_Current'] = pd.to_numeric(df['Time In Title (Current)'], errors='coerce').fillna(0)
    else:
        df['Time_In_Title_Current'] = 0
    
    # Cycles in Level
    df['First_Date_In_Level'] = df.groupby(['Employee_ID','Management_Level_Agg'])['Snapshot_Date'].transform('min')
    df['Years_In_Level'] = (df['Snapshot_Date'] - df['First_Date_In_Level']).dt.days / 365.25
    df['Expected_Years'] = df['Management_Level_Agg'].map(LEVEL_BENCHMARKS).fillna(3.0)
    df['Cycles_In_Level'] = df['Years_In_Level'] / df['Expected_Years']
    df['Is_Stuck_In_Level'] = (df['Cycles_In_Level'] >= 1.5).astype(int)
    
    # Days Since Last Promotion
    if 'Is_Promoted' in df.columns:
        promo_rows = df[df['Is_Promoted'] == 1][['Employee_ID','Snapshot_Date']].copy()
        promo_rows = promo_rows.rename(columns={'Snapshot_Date': 'Last_Promo_Date'})
        promo_rows = promo_rows.sort_values('Last_Promo_Date').drop_duplicates(subset='Employee_ID', keep='last')
        df = pd.merge(df, promo_rows, on='Employee_ID', how='left')
        df['Days_Since_Last_Promotion'] = (df['Snapshot_Date'] - df['Last_Promo_Date']).dt.days
        df['Days_Since_Last_Promotion'] = df['Days_Since_Last_Promotion'].fillna(-1)
        df.drop(columns=['Last_Promo_Date'], inplace=True, errors='ignore')
    else:
        df['Days_Since_Last_Promotion'] = -1
    
    print('   ✅ Tenure & Career features done.')
else:
    print('⏭️ Skipped (Cache active)')

In [ ]:
if rebuild_all:
    print('   [2/6] Performance features...')
    
    if 'Performance_Score' not in df.columns:
        df['Performance_Score'] = np.nan
    df['Is_High_Performer'] = df['Performance_Score'].isin([4, 5]).astype(int)
    
    # YoY Drop
    df['Year_Minus_1'] = df['Year'] - 1
    perf_ref = df[['Employee_ID','Year','Performance_Score']].drop_duplicates(subset=['Employee_ID','Year'])
    perf_ref.columns = ['Employee_ID','Year_Minus_1','Prev_Year_Perf']
    df = pd.merge(df, perf_ref, on=['Employee_ID','Year_Minus_1'], how='left')
    df['Performance_Drop_Flag'] = ((df['Prev_Year_Perf'].notna()) & (df['Performance_Score'] < df['Prev_Year_Perf'])).astype(int)
    df.drop(columns=['Year_Minus_1','Prev_Year_Perf'], inplace=True, errors='ignore')
    
    # Rolling 3yr avg
    yp = df.drop_duplicates(subset=['Employee_ID','Year'])[['Employee_ID','Year','Performance_Score']].copy().sort_values(['Employee_ID','Year'])
    yp['Perf_Score_Rolling_Avg_3Yr'] = yp.groupby('Employee_ID')['Performance_Score'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df = pd.merge(df, yp[['Employee_ID','Year','Perf_Score_Rolling_Avg_3Yr']], on=['Employee_ID','Year'], how='left')
    
    # Trajectory slope
    def calc_slope(series):
        valid = series.dropna()
        if len(valid) < 2: return 0.0
        x = np.arange(len(valid), dtype=float)
        y = valid.values.astype(float)
        xm, ym = x.mean(), y.mean()
        d = ((x - xm) ** 2).sum()
        return ((x - xm) * (y - ym)).sum() / d if d != 0 else 0.0
    
    yps = df.drop_duplicates(subset=['Employee_ID','Year'])[['Employee_ID','Year','Performance_Score']].copy().sort_values(['Employee_ID','Year'])
    yps['Perf_Trajectory_Slope'] = yps.groupby('Employee_ID')['Performance_Score'].transform(lambda x: x.expanding(min_periods=2).apply(calc_slope, raw=False))
    df = pd.merge(df, yps[['Employee_ID','Year','Perf_Trajectory_Slope']], on=['Employee_ID','Year'], how='left')
    df['Perf_Trajectory_Slope'] = df['Perf_Trajectory_Slope'].fillna(0)
    
    # Consecutive low perf years
    yl = df.drop_duplicates(subset=['Employee_ID','Year'])[['Employee_ID','Year','Performance_Score']].copy().sort_values(['Employee_ID','Year'])
    yl['Is_Low'] = (yl['Performance_Score'] <= 2).astype(int)
    yl['Consecutive_Low_Perf_Years'] = yl.groupby('Employee_ID')['Is_Low'].transform(
        lambda x: x * (x.groupby((x != x.shift()).cumsum()).cumcount() + 1)
    )
    df = pd.merge(df, yl[['Employee_ID','Year','Consecutive_Low_Perf_Years']], on=['Employee_ID','Year'], how='left')
    df['Consecutive_Low_Perf_Years'] = df['Consecutive_Low_Perf_Years'].fillna(0)
    
    print('   ✅ Performance features done.')
else:
    print('⏭️ Skipped (Cache active)')

In [ ]:
if rebuild_all:
    print('   [3/6] Peer dynamics...')
    if 'Level 4 Org' not in df.columns: df['Level 4 Org'] = 'Unknown'
    if 'Is_Promoted' not in df.columns: df['Is_Promoted'] = 0
    
    promo_cnt = df.groupby(['Year','Management_Level_Agg','Level 4 Org'])['Is_Promoted'].transform('sum')
    hc_cnt = df.groupby(['Year','Management_Level_Agg','Level 4 Org'])['Employee_ID'].transform('count')
    df['Peer_Promo_Rate'] = promo_cnt / hc_cnt.replace(0, 1)
    df['Peer_Pressure_Flag'] = ((df['Is_Promoted'] == 0) & (df['Peer_Promo_Rate'] > 0.10)).astype(int)
    
    print('   [4/6] Manager intelligence...')
    if 'Manager ID' in df.columns:
        df['Manager ID'] = df['Manager ID'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
        df['Employee_ID'] = df['Employee_ID'].astype(str).str.strip()
        if 'Active Manager Direct EMP+CWK Reports' in df.columns:
            df['Manager_Span_Of_Control'] = pd.to_numeric(df['Active Manager Direct EMP+CWK Reports'], errors='coerce').fillna(0)
        else:
            df['Manager_Span_Of_Control'] = df.groupby(['Manager ID','Snapshot_Date'])['Employee_ID'].transform('count')
        df['Manager_Past_Churn_Count'] = 0
        mgr_perf = df[['Employee_ID','Snapshot_Date','Performance_Score']].copy()
        mgr_perf.columns = ['Manager ID','Snapshot_Date','Manager_Performance_Score']
        mgr_perf['Manager ID'] = mgr_perf['Manager ID'].astype(str).str.strip()
        mgr_perf = mgr_perf.drop_duplicates(subset=['Manager ID','Snapshot_Date'])
        try:
            df = pd.merge(df, mgr_perf, on=['Manager ID','Snapshot_Date'], how='left')
        except:
            df['Manager_Performance_Score'] = np.nan
        df['Manager_Performance_Score'] = df['Manager_Performance_Score'].fillna(3)
        df['Performer_vs_Mgr_Score'] = df['Performance_Score'] / df['Manager_Performance_Score'].replace(0, 1)
        df.loc[df['Performance_Score'].isna(), 'Performer_vs_Mgr_Score'] = np.nan
        df['Prev_Mgr'] = df.groupby('Employee_ID')['Manager ID'].shift(1)
        df['Mgr_Change'] = ((df['Prev_Mgr'].notna()) & (df['Manager ID'] != df['Prev_Mgr'])).astype(int)
        df['Manager_Changed_Recently'] = df.groupby('Employee_ID')['Mgr_Change'].transform(
            lambda x: x.rolling(window=6, min_periods=1).max()
        ).fillna(0).astype(int)
        df.drop(columns=['Prev_Mgr','Mgr_Change'], inplace=True, errors='ignore')
    else:
        for c in ['Manager_Past_Churn_Count','Manager_Performance_Score','Performer_vs_Mgr_Score','Manager_Changed_Recently','Manager_Span_Of_Control']:
            df[c] = 0
    
    print('   ✅ Peer & Manager features done.')
else:
    print('⏭️ Skipped (Cache active)')

In [ ]:
if rebuild_all:
    print('   [5/6] Org-level risk features...')
    if 'Job Family' in df.columns:
        jf = df.drop_duplicates(subset=['Year','Job Family'])[['Year','Job Family',TARGET_COLUMN]].copy()
        jf.columns = ['Prev_Year_JF','Job Family','Job_Family_Churn_Rate']
        jf['Prev_Year_JF'] = jf['Prev_Year_JF'] + 1
        df = pd.merge(df, jf, left_on=['Year','Job Family'], right_on=['Prev_Year_JF','Job Family'], how='left')
        df['Job_Family_Churn_Rate'] = df['Job_Family_Churn_Rate'].fillna(0)
        df.drop(columns=['Prev_Year_JF'], inplace=True, errors='ignore')
    else:
        df['Job_Family_Churn_Rate'] = 0
    
    org = df.drop_duplicates(subset=['Year','Level 4 Org'])[['Year','Level 4 Org',TARGET_COLUMN]].copy()
    org.columns = ['Prev_Year_Org','Level 4 Org','Org_Level_Churn_Rate']
    org['Prev_Year_Org'] = org['Prev_Year_Org'] + 1
    df = pd.merge(df, org, left_on=['Year','Level 4 Org'], right_on=['Prev_Year_Org','Level 4 Org'], how='left')
    df['Org_Level_Churn_Rate'] = df['Org_Level_Churn_Rate'].fillna(0)
    df.drop(columns=['Prev_Year_Org'], inplace=True, errors='ignore')
    
    print('   [6/6] Context & temporal features...')
    df['Is_Lateral_Hire'] = pd.to_numeric(df.get('HC_Hire_Lateral_Comm', 0), errors='coerce').fillna(0).abs().clip(0,1).astype(int)
    df['Is_Campus_Hire'] = pd.to_numeric(df.get('HC_Hire_Campus_Comm', 0), errors='coerce').fillna(0).abs().clip(0,1).astype(int)
    
    if 'International Assignment Flag' in df.columns:
        df['Has_International_Assignment'] = df['International Assignment Flag'].astype(str).str.lower().isin(['yes','y','1','true']).astype(int)
    else:
        df['Has_International_Assignment'] = 0
    
    if 'Acquisition Company' in df.columns:
        df['Is_Acquisition_Employee'] = (df['Acquisition Company'].notna() & (df['Acquisition Company'].astype(str).str.strip() != '') & (df['Acquisition Company'].astype(str).str.lower() != 'nan')).astype(int)
    else:
        df['Is_Acquisition_Employee'] = 0
    
    df['FTE_Value'] = pd.to_numeric(df.get('FTE', 1.0), errors='coerce').fillna(1.0)
    
    for rc in ['HC_RGI_EE','HC_RGO_EE']:
        if rc not in df.columns: df[rc] = 0
    df['_re'] = ((df['HC_RGI_EE']==1) | (df['HC_RGO_EE']==1)).astype(int)
    df['Is_Reorg_Affected'] = df.groupby('Employee_ID')['_re'].transform(lambda x: x.rolling(6, min_periods=1).max()).fillna(0).astype(int)
    
    for xc in ['HC_XIN_EE','HC_XOT_EE']:
        if xc not in df.columns: df[xc] = 0
    df['_xf'] = ((df['HC_XIN_EE']==1) | (df['HC_XOT_EE']==1)).astype(int)
    df['Is_Transfer_Recent'] = df.groupby('Employee_ID')['_xf'].transform(lambda x: x.rolling(6, min_periods=1).max()).fillna(0).astype(int)
    df.drop(columns=['_re','_xf'], inplace=True, errors='ignore')
    
    df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    df['Is_Post_Bonus_Window'] = df['Month'].isin([2, 3, 4]).astype(int)
    
    # Fill missing features
    for feat in BASE_FEATURES:
        if feat not in df.columns:
            df[feat] = 0
        if df[feat].dtype in ['float64','float32','int64','int32']:
            df[feat] = df[feat].fillna(0)
    
    print(f'\n   ✅ All {len(BASE_FEATURES)} features built. Shape: {df.shape}')
else:
    print('⏭️ Skipped (Cache active)')

---
## 5. Save Featured Data & Prepare Model Matrices

In [ ]:
if rebuild_all:
    # Save for Dashboard notebook
    feat_path = os.path.join(PATH_OUTPUT, 'Model_Ready_Data.parquet')
    try:
        df.to_parquet(feat_path, index=False)
        print(f'✅ Featured data saved: {feat_path}')
    except Exception as e:
        feat_path = os.path.join(PATH_OUTPUT, 'Model_Ready_Data.csv')
        df.to_csv(feat_path, index=False)
        print(f'✅ Featured data saved as CSV: {feat_path}')
    
    # Prepare model matrices
    # Auto-select the latest year with positive targets for validation
    years_pos = df.groupby('Year')[TARGET_COLUMN].sum()
    years_with = years_pos[years_pos > 0]
    if not years_with.empty:
        PRED_YEAR = int(years_with.index.max())
        print(f'\n   Auto-selected prediction year: {PRED_YEAR} ({int(years_with[PRED_YEAR])} positive snapshots)')
    else:
        PRED_YEAR = df['Year'].max()
        print(f'   WARNING: No year has positives. Using {PRED_YEAR}')
    
    all_features = BASE_FEATURES + CATEGORICAL_FEATURES
    available = [f for f in all_features if f in df.columns]
    
    X_all = pd.get_dummies(df[available], columns=CATEGORICAL_FEATURES, drop_first=True)
    y_all = df[TARGET_COLUMN]
    
    train_mask = df['Year'] < PRED_YEAR
    X_train = X_all[train_mask].fillna(0)
    y_train = y_all[train_mask].fillna(0)
    train_emp_ids = df.loc[train_mask, 'Employee_ID']
    
    test_raw = df[df['Year'] == PRED_YEAR].sort_values(['Employee_ID','Snapshot_Date'], ascending=[True, False])
    test_unique = test_raw.drop_duplicates(subset=['Employee_ID'], keep='first')
    X_test = pd.get_dummies(test_unique[available], columns=CATEGORICAL_FEATURES, drop_first=True)
    y_test = test_unique[TARGET_COLUMN]
    for c in X_train.columns:
        if c not in X_test.columns: X_test[c] = 0
    X_test = X_test[X_train.columns].fillna(0)
    feat_names = X_train.columns.tolist()
    
    print(f'   Train: {len(X_train):,} rows ({y_train.sum():,.0f} positive)')
    print(f'   Test:  {len(X_test):,} rows ({y_test.sum():,.0f} positive)')
    print(f'   Unique employees in train: {train_emp_ids.nunique():,}')
else:
    print('⏭️ Skipped (Cache active)')

---
## 6. Train XGBoost Model

In [ ]:
if rebuild_all:
    from xgboost import XGBClassifier
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, confusion_matrix
    
    try:
        from imblearn.over_sampling import SMOTE
        HAS_SMOTE = True
    except ImportError:
        HAS_SMOTE = False
        print('   imblearn not available. Skipping SMOTE.')
    
    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    scale_pos = n_neg / max(n_pos, 1)
    
    xgb_params = {
        'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'min_child_weight': 5, 'gamma': 0.1,
        'scale_pos_weight': scale_pos,
        'eval_metric': 'auc', 'random_state': 42, 'use_label_encoder': False,
    }
    
    # Apply SMOTE to full training set
    if HAS_SMOTE and n_pos > 5:
        smote = SMOTE(random_state=42, k_neighbors=min(5, int(n_pos) - 1))
        X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
        print(f'   SMOTE: {len(X_train):,} -> {len(X_train_res):,} rows')
    else:
        X_train_res, y_train_res = X_train, y_train
    
    print('\n🚀 Training XGBoost...')
    start = time.time()
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(X_train_res, y_train_res, verbose=False)
    print(f'✅ Model trained in {time.time()-start:.1f}s')
else:
    print('⏭️ Skipped (Cache active)')

In [ ]:
if rebuild_all:
    # GroupKFold Cross-Validation (no employee leakage)
    print('\n📊 GroupKFold Cross-Validation (5 folds)...')
    gkf = GroupKFold(n_splits=5)
    auc_scores, recall_scores, prec_scores = [], [], []
    
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=train_emp_ids), 1):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        if y_val.sum() == 0 or y_tr.sum() == 0:
            continue
        if HAS_SMOTE and y_tr.sum() > 5:
            try:
                sm = SMOTE(random_state=42, k_neighbors=min(5, int(y_tr.sum())-1))
                X_tr, y_tr = sm.fit_resample(X_tr, y_tr)
            except: pass
        m = XGBClassifier(**xgb_params)
        m.fit(X_tr, y_tr, verbose=False)
        yp = m.predict_proba(X_val)[:,1]
        ypd = (yp > 0.5).astype(int)
        try: auc_scores.append(roc_auc_score(y_val, yp))
        except: auc_scores.append(0)
        recall_scores.append(recall_score(y_val, ypd, zero_division=0))
        prec_scores.append(precision_score(y_val, ypd, zero_division=0))
        print(f'   Fold {fold}: AUC={auc_scores[-1]:.4f} Recall={recall_scores[-1]:.4f} Prec={prec_scores[-1]:.4f}')
    
    print(f'\n   CV AUC:       {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}')
    print(f'   CV Recall:    {np.mean(recall_scores):.3f} ± {np.std(recall_scores):.3f}')
    print(f'   CV Precision: {np.mean(prec_scores):.3f} ± {np.std(prec_scores):.3f}')
else:
    print('⏭️ Skipped (Cache active)')

---
## 7. Evaluate on Test Set

In [ ]:
if rebuild_all:
    threshold = 0.5
    y_prob = xgb_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob > threshold).astype(int)
    
    test_auc = roc_auc_score(y_test, y_prob) if y_test.sum() > 0 else 0
    test_recall = recall_score(y_test, y_pred, zero_division=0)
    test_prec = precision_score(y_test, y_pred, zero_division=0)
    test_f1 = f1_score(y_test, y_pred, zero_division=0)
    
    print('=' * 50)
    print(f'   TEST SET RESULTS ({PRED_YEAR})')
    print('=' * 50)
    print(f'   AUC:       {test_auc:.4f}')
    print(f'   Recall:    {test_recall:.4f}')
    print(f'   Precision: {test_prec:.4f}')
    print(f'   F1:        {test_f1:.4f}')
    print(f'   Population: {len(X_test):,}  |  Actual leavers: {int(y_test.sum()):,}  |  Flagged: {int(y_pred.sum()):,}')
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
    axes[0].set_title(f'Confusion Matrix ({PRED_YEAR})')
    axes[1].hist(y_prob[y_test==0], bins=50, alpha=0.6, label='Stayers', color='teal')
    axes[1].hist(y_prob[y_test==1], bins=50, alpha=0.6, label='Leavers', color='salmon')
    axes[1].axvline(threshold, color='red', linestyle='--', label=f'Threshold {threshold}')
    axes[1].set_title('Risk Distribution'); axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print('⏭️ Skipped (Cache active)')

In [ ]:
if rebuild_all:
    # Feature Importance
    imps = pd.DataFrame({'Feature': feat_names, 'Importance': xgb_model.feature_importances_}).sort_values('Importance', ascending=False).head(15)
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=imps, palette='viridis')
    plt.title('Top 15 Drivers of Attrition')
    plt.tight_layout()
    plt.show()
else:
    print('⏭️ Skipped (Cache active)')

---
## 8. Save Model & Artifacts

In [ ]:
if rebuild_all:
    # Save model
    model_path = os.path.join(PATH_MODEL_OUTPUT, 'xgboost_model.joblib')
    joblib.dump(xgb_model, model_path)
    print(f'✅ Model saved: {model_path}')
    
    # Save feature names
    feat_path = os.path.join(PATH_MODEL_OUTPUT, 'feature_names.json')
    with open(feat_path, 'w') as f:
        json.dump(feat_names, f)
    print(f'✅ Feature names saved: {feat_path}')
    
    # Save params
    params_path = os.path.join(PATH_MODEL_OUTPUT, 'model_params.json')
    with open(params_path, 'w') as f:
        json.dump({k: str(v) for k, v in xgb_params.items()}, f, indent=2)
    print(f'✅ Model params saved: {params_path}')
    
    print(f'\n🎉 Pipeline complete! Open the Dashboard notebook to see results.')
else:
    print('✅ Notebook finished without changes. Open the Dashboard notebook to see results.')